In [ ]:
#!pip install statsbombpy
#!pip install pandas
#!pip install networkx

: 

In [ ]:
import statsbombpy
from statsbombpy import sb
import pandas as pd

In [ ]:
matches = sb.matches(competition_id=2, season_id=27)
matches.head()

In [ ]:
# Get the first match ID
match_id = matches.iloc[0]['match_id']

# Load event data
events = sb.events(match_id=match_id)
events.head()

In [ ]:
all_events = []
for match_id in matches['match_id']:
    events = sb.events(match_id=match_id)
    events['match_id'] = match_id
    all_events.append(events)

# Combine into one big DataFrame
pl_2015_2016_events = pd.concat(all_events, ignore_index=True)

In [ ]:
pl_2015_2016_events.head()

In [ ]:
leicester_matches = matches[(matches['home_team'] == "Leicester City") | 
                            (matches['away_team'] == "Leicester City")]

In [ ]:
events_list = []
for match_id in leicester_matches['match_id']:
    df = sb.events(match_id=match_id)
    events_list.append(df)

events = pd.concat(events_list, ignore_index=True)

In [ ]:
import numpy as np

In [ ]:
# copy to avoid SettingWithCopyWarning and extract X coordinate from location
lec_events = events[events['team'] == 'Leicester City'].copy()
lec_events['x'] = lec_events['location'].apply(
    lambda loc: loc[0] if isinstance(loc, (list, tuple)) and len(loc) >= 1 else np.nan
)

In [ ]:
# filter to attacking third (x between 80 and 120)
att_third = lec_events[(lec_events['x'] >= 80) & (lec_events['x'] <= 120)].copy()

In [ ]:
# mark unsuccessful dribbles where type == 'Dribble' and dribble_outcome == 'Incomplete'
att_third['unsuccessful_dribble'] = (
    (att_third['type'] == 'Dribble') & (att_third['dribble_outcome'] == 'Incomplete')
).astype(int)

# aggregate per player using only attacking-third events
possession_ending_actions = att_third.groupby('player').agg(
    shots=('type', lambda x: (x == 'Shot').sum()),
    dispossessed=('type', lambda x: (x == 'Dispossessed').sum()),
    miscontrols=('type', lambda x: (x == 'Miscontrol').sum()),
    incomplete_passes=('pass_outcome', lambda x: x.notna().sum()),
    unsuccessful_dribbles=('unsuccessful_dribble', 'sum')
).reset_index()

In [ ]:
cols = ['shots','dispossessed','miscontrols','incomplete_passes','unsuccessful_dribbles']
possession_ending_actions[cols] = possession_ending_actions[cols].fillna(0).astype(int)
possession_ending_actions['possession_ending_actions'] = possession_ending_actions[cols].sum(axis=1)


In [ ]:
possession_ending_actions

In [ ]:
# find possession_ending actions for Leceister City as a team
shots = (att_third['type'] == 'Shot').sum()
dispossessed = (att_third['type'] == 'Dispossessed').sum()
miscontrols = (att_third['type'] == 'Miscontrol').sum()
incomplete_passes = att_third['pass_outcome'].notna().sum()
unsuccessful_dribbles = att_third['unsuccessful_dribble'].sum()

In [ ]:
team_totals = pd.DataFrame([{
    'team': 'Leicester City',
    'shots': shots,
    'dispossessed': dispossessed,
    'miscontrols': miscontrols,
    'incomplete_passes': incomplete_passes,
    'unsuccessful_dribbles': unsuccessful_dribbles,
    'possession ending actions': (shots + dispossessed + miscontrols + incomplete_passes + unsuccessful_dribbles)
}])
team_totals

# Change to per90 numbers
# Add chances created per 90 - find G/Aper90 as well, so we can compare usage rate to these numbers - look at wyscout

In [ ]:
duel_values = lec_events['duel_outcome'].value_counts(dropna=False)
duel_values

In [ ]:
type_values = lec_events['type'].value_counts(dropna=False)
type_values

# find values for when 'type' equals Dribble
dribble_values = lec_events[lec_events['type'] == 'Dribble']
dribble_values['dribble_outcome'].value_counts(dropna=False)

In [ ]:
# create usage rate by dividing player possession ending actions by team total, and display usage rate with two decimal places
possession_ending_actions['Usage Rate'] = ((possession_ending_actions['possession_ending_actions'] / team_totals['possession ending actions'].values[0]) * 100).round(2)

In [ ]:
possession_ending_actions 

In [ ]:
!pip install statsbombpy
!pip install pandas
!pip install networkx
!pip install soccerdata
!pip install setuptools
!pip install pyarrow
!pip install polars

In [ ]:
import sys, subprocess

print("Python executable:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "show", "soccerdata"])

In [ ]:
!pip install soccerdata

In [ ]:
import soccerdata as sd